## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 12 — A pipeline whose stages read different sources

The matrix analog of notebook 07, and the notebook 07 could never actually
be.

07's original design had stage 1 read one file per worker and stage 2 read
a *different* file — the processed output of stage 1. That was impossible
under the old harness: `CSV_PATH` was a single per-container environment
variable fixed at deploy time, so a worker could only ever see one file,
and no skill took a path parameter.

Now a worker holds **several named sources** and chooses between them
itself, based on what the stage is asking for. The head still names
nothing. This is the first notebook where `choose_source` has more than
one candidate to pick from, which is the part being tested.

Each worker holds two sources:

- **raw** — uncalibrated sensor readings straight off the equipment
- **calibrated** — the same measurements after a per-unit correction

The pipeline computes statistics over each, then compares them, so the
answer is only right if each stage reached the source it was asking for.


## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4,worker5").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : Gustavo


## Generate each worker's private data

One CSV per worker with synthetic sensor data for pipeline processing.

In [3]:
import numpy as np
import pandas as pd
import yaml

DATA_DIR = os.environ.get("DATA_DIR", "/data")
ROWS = 60

# Two sources per worker, at unrelated paths, with names that say what
# they are rather than where they live or whose they are.
LAYOUT = {
    "worker1": ("intake/line_a_raw.csv",      "line_a_uncalibrated",
                "vault/line_a_corrected.csv", "line_a_calibrated"),
    "worker2": ("intake/line_b_raw.csv",      "line_b_uncalibrated",
                "vault/line_b_corrected.csv", "line_b_calibrated"),
    "worker3": ("spool/raw_capture.csv",      "spool_uncalibrated",
                "vault/spool_corrected.csv",  "spool_calibrated"),
    "worker4": ("dropbox/unverified.csv",     "dropbox_uncalibrated",
                "vault/dropbox_corrected.csv","dropbox_calibrated"),
}

# The calibration is a known, fixed offset and gain per worker, so the
# expected relationship between the two sources is exactly computable.
CALIBRATION = {"worker1": (1.00, 2.0), "worker2": (1.00, 2.0),
               "worker3": (1.00, 2.0), "worker4": (1.00, 2.0)}

for i, (node, (raw_rel, raw_name, cal_rel, cal_name)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=4200 + i)
    start = pd.Timestamp("2026-01-01T00:00:00")
    ts = [start + pd.Timedelta(minutes=k) for k in range(ROWS)]
    raw_vals = np.round(rng.normal(50.0 + 3 * i, 5.0, ROWS), 4)
    gain, offset = CALIBRATION[node]
    cal_vals = np.round(raw_vals * gain + offset, 4)

    specs = [
        (raw_rel, raw_name, raw_vals,
         "Uncalibrated sensor readings, straight off the equipment, no correction applied."),
        (cal_rel, cal_name, cal_vals,
         "The same measurements after the per-unit calibration correction has been applied."),
    ]
    for rel, name, vals, desc in specs:
        path = os.path.join(DATA_DIR, rel)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pd.DataFrame({"timestamp": ts, "reading": vals}).to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [
            {"name": name, "type": "csv", "mode": "local",
             "description": f"{desc} {ROWS} rows. Numeric column: reading.",
             "path": os.path.join(DATA_DIR, rel)}
            for rel, name, vals, desc in specs
        ]}, fh, sort_keys=False)

    print(f"{node}: {raw_name} and {cal_name}  ({ROWS} rows each)")

# Ground truth for the comparison the head will be asked to make.
_raw_all, _cal_all = [], []
for i, (node, (raw_rel, _, cal_rel, _)) in enumerate(LAYOUT.items(), start=1):
    _raw_all.append(pd.read_csv(os.path.join(DATA_DIR, raw_rel))["reading"].values)
    _cal_all.append(pd.read_csv(os.path.join(DATA_DIR, cal_rel))["reading"].values)
_raw_all = np.concatenate(_raw_all); _cal_all = np.concatenate(_cal_all)
print(f"\nexpected: raw mean {_raw_all.mean():.4f}, calibrated mean "
      f"{_cal_all.mean():.4f}, difference {_cal_all.mean()-_raw_all.mean():.4f}")


worker1: line_a_uncalibrated and line_a_calibrated  (60 rows each)
worker2: line_b_uncalibrated and line_b_calibrated  (60 rows each)
worker3: spool_uncalibrated and spool_calibrated  (60 rows each)
worker4: dropbox_uncalibrated and dropbox_calibrated  (60 rows each)

expected: raw mean 57.9117, calibrated mean 59.9117, difference 2.0000


## Bring the fleet up

In [4]:
AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

In [5]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask the LLM head to orchestrate the pipeline

In this simplified version, each worker has its own CSV with all needed data,
and the head coordinates which workers run which stage. The head can decide
to use `sum` with different transforms across the workers' CSV data.

In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# Each sum call carries a different `objective`, and that objective is the
# ONLY thing steering which of its two sources a worker reads. The head
# never names either one - it does not know them, and does not need to.
#
# This is the case CSV_PATH made impossible: one worker, two files, two
# stages of the same pipeline reaching different ones.
answer = await ask_scarlet_agent(
    "Compare uncalibrated against calibrated sensor readings across the "
    "fleet. Each worker holds both; you must not name any source yourself. "
    "Stage 1: call sum with transform=identity and "
    "objective='uncalibrated raw sensor readings with no correction applied'. "
    "Stage 2: call sum with transform=identity and "
    "objective='calibrated sensor readings after the correction was applied'. "
    "Stage 3: use combine to compute each stage's mean as its sum divided "
    "by its n, then the difference between the two means. "
    "Report the raw mean, the calibrated mean, and the difference. State "
    "the n you used."
)
print(answer)


The comparison between uncalibrated and calibrated sensor readings yields the following results:

- **Uncalibrated raw mean**: 13898.8007 / 240 = **57.9117**
- **Calibrated mean**: 14378.8007 / 240 = **59.9117**
- **Difference (uncalibrated − calibrated)**: **-2.0**
- **n (number of readings)**: **240**

This shows that, on average, the calibrated readings are **2.0 units higher** than the uncalibrated readings.


## Teardown

In [7]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
